# Urban Growth Prediction

## Import necessary libraries

In [1]:
# import libraries
import pandas as pd
import ee
import geemap

In [2]:
# Authenticate GEE
ee.Authenticate()

True

In [ ]:
# Initialize GEE
ee.Initialize()

## Centre of AOI - FCT Abuja

In [ ]:
# Center coordinate to show map
fct_center = (9.056266, 7.498522)

## Visualization Parameters

In [ ]:
# Boundary visualization params
vis_params_fao_1 = {
    "fillcolor":"b5ffb4", "color": "00909f",
    "width":1.0,
}
vis_params_aoi = {"fillcolor":"","color":"red"}
# Sentinel-2 visualization parameters
vis_params_s2_rgb = {"min":0, "max":0.3,"bands":["B4", "B3", "B2"]}
vis_params_s2_fcc = {"min":0, "max":0.3,"bands":["B8", "B4", "B3"]}

# Spectral indices visualization parameters
# NDVI
vis_params_ndvi = {
    "min":-0.2,
    "max" : 0.8,
    "palette":[
        "#a50026",
        "#d73027",
        "#f46d43",
        "#fdae61",
        "#fee08b",
        "#d9ef8b",
        "#86d96a",
        "#66bd63",
        "#1a9850",
        "#006837",
    ],
}




# NDWI
vis_params_ndwi = {
    "min":-0.5,
        "max": 0.5,
        "palette": [
            "#543005",
            "#8c510a",
            "#d8b365",
            "#f6e863",
            "#c7eae5",
            "#5ab4ac",
            "#01665e",


        ]

}
# NDBI

vis_params_ndbi = {
    "min": -0.5,
    "max": 0.5,
    "palette": [
        "#ffffff",  
        "#ffffff",  
        "#f0f4f8",  
        "#d9e6f2",  
        "#99badd",  
        "#4a7cb5",  
        "#0f4c81"   
    ]
}

In [ ]:

vis_params_Fao_1 = {
  "fillColor": 'b5ffb4',
  "color": '00909F',
  "width": 1.0,
}
vis_params_aoi =  {"fillColor":"", "color": "red"}
# Boundary visualization params 
vis_params_fao_1 = {
  "fillColor": 'b5ffb4',
  "color": '00909F',
  "width": 1.0,
}

vis_params_aoi = {"fillcolor": "", "color": "red"}

# Sentinel-2 Visualization parameters 
vis_params_s2_rgb = {"min" : 0, "max" :0.3, "bands": ["B4", "B3", "B2"]}
vis_params_s2_fcc = {"min" : 0, "max" :0.3, "bands": ["B8", "B4", "B3"]}



# Visualisation parameters for road layers
vis_params_roads_vector = {
                        "color": "red",
                        "width": 1.5,
                    }

vis_params_roads_raster = {
                        "min": 0,
                        "max": 1,
                        "palette": ["black", "white"],
                    }


vis_params_dist_road = {
    "min": 0,
    "max": 21730, #meters
    "palette": [
        "#d73027",  
        "#f46d43",
        "#fdae61",
        "#fee08b",
        "#d9ef8b",
        "#a6d96a",
        "#1a9850",  
    ],
}

# Visualisation parameters for water layers
vis_params_water = {
    "min": 0,
    "max": 1,
    "palette": ["white", "blue"], 
}

# Distance to water
vis_params_dist_water = {
    "min": 0,
    "max": 38497,  
    "palette": [
        "#f7fbff",  
        "#deebf7",
        "#9ecae1",
        "#4292c6",
        "#2171b5",
        "#08306b",  
    ],
}


# Nighttime lights visualisation parameters
vis_params_ntl = {
    "min": 0,
    "max": 20,  
    "palette": [
        "#000000",  
        "#2c0b00",
        "#6e1c00",
        "#a83800",
        "#d9720a",
        "#f7b733",
        "#ffe98a",  
    ],
}


# Visualization parameters for GPWv411 population DENSITY layer
vis_params_gpw = {
  "min": 0.0,
  "max": 10000.0,
  "palette": ["ffffe7", "FFc869", "ffac1d", "e17735", "f2552c", "9f0c21"]
  }

# Nigeria Boundary

In [ ]:
# Boundary Data From FAO GAUL
# Data source: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_SIMPLIFIED_500m_2015_level0
fao_gaul_l0 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level0') # Countries boundaries
fao_gaul_l1 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level1') # States boundaries
fao_gaul_l2 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level2') # LGAs boundaries





boundary_Map = geemap.Map(center=fct_center, zoom=10)
boundary_Map.addLayer(fao_gaul_l0, {}, 'Country Boundaries')
boundary_Map.addLayer(fao_gaul_l1, {}, 'State Boundaries')
boundary_Map.addLayer(fao_gaul_l2, {}, 'LGA Boundaries')
boundary_Map

## Abuja Boundary

In [ ]:
# Select just a single feature
print(fao_gaul_l0.limit(1).getInfo()["columns"])
nga_l0 = fao_gaul_l0.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
nga_l1 = fao_gaul_l1.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
fct_l0 = nga_l1.filter(ee.Filter.eq("ADM1_NAME", "Abuja"))
print(nga_l0.getInfo())
#get geometry of Abuja Boundary Feature Collection
aoi = fct_l0.geometry()
aoi_bbox = aoi.bounds()

#Create a map to visualize Abuja Boundary
aoi_map = geemap.Map(center=fct_center, zoom=10)


aoi_map.addLayer(fct_l0, vis_params_aoi, 'Abuja Boundary')
aoi_map


## Sentinel 2 Processing & Indices Function

In [ ]:

def process_sentinel2_composite(aoi, start_date, end_date):
    """
    Filters, scales, computes median composite, clips to AOI, 
    and adds spectral indices (NDVI, NDWI, NDBI).
    """
    # 1. Filter Sentinel-2 Collection
    s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterDate(start_date, end_date)
              .filterBounds(aoi)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
              .map(lambda img: img.multiply(0.0001).copyProperties(img, ["system:time_start"])))
    
    # 2. Compute Median Composite and Clip to AOI
    s2_composite = s2_col.median().clip(aoi)
    
    # 3. Calculate Spectral Indices
    ndvi = s2_composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = s2_composite.normalizedDifference(['B3', 'B8']).rename('NDWI')
    ndbi = s2_composite.normalizedDifference(['B11', 'B8']).rename('NDBI')
    
    # Return composite with added index bands
    return s2_composite.addBands([ndvi, ndwi, ndbi])

# Execute Function for Study Extent
aoi = fct_l0.geometry()
training_image = process_sentinel2_composite(aoi, '2022-01-01', '2022-12-31')

# Print bands to confirm NDVI, NDWI, and NDBI are present
print("Processed Image Bands:", training_image.bandNames().getInfo())

# Map Visualization
vis_params_rgb = {"min": 0, "max": 0.3, "bands": ["B4", "B3", "B2"]}
S2_map1 = geemap.Map(center=fct_center, zoom=9)
S2_map1.addLayer(fct_l0, vis_params_aoi, 'FCT Boundary')
S2_map1.addLayer(training_image, vis_params_rgb, 'S2 Composite (RGB)')
S2_map1.addLayer(training_image.select('NDVI'), {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'green']}, 'NDVI')
S2_map1

## Random Points Sampling

In [ ]:
def generate_random_points(image, region, num_points=500, scale=30):
    """
    Generates random spatial points and extracts pixel values safely without exceeding memory limits.
    """
    # 1. Filter image to essential spectral bands & indices to lighten memory
    light_image = image.select(['B4', 'B3', 'B2', 'NDVI', 'NDWI', 'NDBI'])
    
    # 2. Generate random points
    random_pts = ee.FeatureCollection.randomPoints(region=region, points=num_points, seed=42)
    
    # 3. Extract pixel values using tileScale=4 or 8 to prevent memory overflow
    sampled_points = light_image.sampleRegions(
        collection=random_pts,
        scale=scale,
        tileScale=4,  # Splits processing into smaller tiles to avoid memory limit
        geometries=True
    )
    return sampled_points

# Run optimized sampling
random_samples = generate_random_points(
    image=training_image,
    region=aoi,
    num_points=500,
    scale=30
)

# Print count safely
#print("Sampled Points Count:", random_samples.size().getInfo())

In [ ]:
# Data source :
# Image collection
S2_img_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') # All sentinel-2 Collections
    .filterDate('2022-01-01', '2022-01-31') # Limit to January 2022
    .filterBounds(fct_l0.geometry()) # Limit to Abuja
    # Pre-filter to get less cloudy granules.
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    
)
# Apply scale factor
def apply_scale(image):
    scaled_image = image.multiply(0.0001).copyProperties(image,["system:time_start"])
    return scaled_image
S2_img_col = S2_img_col.map(apply_scale)
# Collection Properties
#print(f"Number of images in S2 collection: {S2_img_col.size().getInfo()}\n")
# First image in the collection
first_S2_img = S2_img_col.first()
#print(f"first image in S2 collection: {first_S2_img.getInfo()}\n")

# Bands in S2 image
#print(first_S2_img.bandNames().getInfo())

# Select just band B4
Red_band_s2 = first_S2_img.select(['B4','B3','B2'])
#print(f"Red band: {Red_band_s2.bandNames().getInfo()}")

# Visualize S2 image
vis_params_S2 = {"min" : 0, "max": 0.3, "bands": ["B8", "B4", "B3"]}
vis_params_S2_rbg = {"min" : 0, "max": 0.3, "bands": ["B4", "B3", "B2"]}
S2_map1 = geemap.Map(center= fct_center, zoom=8)
S2_map1.addLayer(fct_l0, vis_params_aoi,"FCT Boundary")
S2_map1.addLayer(first_S2_img.clip(aoi), vis_params_S2_rbg, 'First S2 Image')
# Mosaic the entire S2 collection & clip to FCT extent [Spatial Mosaic]
S2_mosaic = S2_img_col.mosaic()
S2_mosaic_clipped = S2_mosaic.clip(aoi)


#print("Mosaiced S2 Bands:", S2_mosaic_clipped.bandNames().getInfo())

# Add layer to the map
S2_map1.addLayer(S2_mosaic_clipped, vis_params_S2_rbg, 'S2 Mosaiced - Spatial')


S2_map1


In [ ]:
# Compute median composite the entire S2 collection and clip to FCT extent [Temporal Composite]
S2_median = S2_img_col.median().clip(aoi)
#print(f"Median of S2 collection:{S2_median.getInfo()}")

# Compute spectral indices
ndvi = S2_median.normalizedDifference(["B8", "B4"]).rename("ndvi")

ndwi = S2_median.normalizedDifference(["B3", "B8"]).rename("ndwi")
ndbi = S2_median.normalizedDifference(["B8", "B12"]).rename("ndbi")
S2_map3 = geemap.Map(center= fct_center, zoom=8)
S2_map3.addLayer(fct_l0, vis_params_aoi,"FCT Boundary")
S2_map3.addLayer(S2_median, vis_params_s2_fcc, "S2 Median Comp")
S2_map3.addLayer(ndvi, vis_params_ndvi, "S2 NDVI")
S2_map3.addLayer(ndwi, vis_params_ndwi, "S2 NDWI")
S2_map3.addLayer(ndbi, vis_params_ndbi, "S2 NDBI")


S2_map3

## Elevation and Slope

In [ ]:
# Download elevation and compute slope
DEM = ee.Image('USGS/SRTMGL1_003')
DEM.bandNames().getInfo()
elevation = DEM.select('elevation')
slope = ee.Terrain.slope(elevation)
DEM_map = geemap.Map(center = fct_center, zoom=8)
DEM_map.add_basemap('SATELLITE')

DEM_map.addLayer(DEM.clip(fct_l0.geometry()), {'min': 0, 'max': 300, 'palette': ['red', 'green', 'yellow']}, 'DEM')
DEM_map.addLayer(slope.clip(fct_l0.geometry()), {'min': 0, 'max': 10, 'palette': ['red', 'green', 'yellow']}, 'Slope')
DEM_map


# Distance to road

In [ ]:
# Distance to roads
# Data source:https://gee-community-catalog.org/projects/grip/?h=road
roads_africa = ee.FeatureCollection("projects/sat-io/open-datasets/GRIP4/Africa")

# Roads that intersect with study extent
roads_aoi = roads_africa.filterBounds(fct_l0.geometry())

# Convert roads from vector to raster
roads_raster = ee.Image().float().paint(roads_aoi, 1).clip(fct_l0.geometry()) # Where there are roads = 1, else = 0

distance_to_roads = (
    roads_raster.fastDistanceTransform(256)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt()) #convert to meters
    .rename('distance_to_roads')
    .clip(fct_l0.geometry()
) )

# Maximum distance 'distance_to_roads' layer
print(distance_to_roads.reduceRegion(ee.Reducer.max(), fct_l0.geometry(), 1000, maxPixels=1e9).getInfo())
thematic_map_1 = geemap.Map(center=fct_center, zoom=8)
thematic_map_1.add_basemap('SATELLITE')
thematic_map_1.addLayer(roads_raster, vis_params_roads_raster, 'Roads Raster')
thematic_map_1.addLayer(distance_to_roads.select('distance_to_roads'), vis_params_dist_road, 'Distance to Roads')
thematic_map_1.addLayer(roads_aoi, vis_params_roads_vector, 'Roads Vector')
thematic_map_1


# Distance to water

In [ ]:
# Permanent water (JRC Global Surface Water)
# Data source: https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_GlobalSurfaceWater
gsw = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').clip(fct_l0.geometry())
permanent_water = gsw.select('occurrence').gte(50) # >= 50% the time the water present

# Compute Euclidean distance to permanent water
distance_to_water = (
    permanent_water.Not()
    .fastDistanceTransform(256)
    .sqrt()
    .multiply(ee.Image.pixelArea().sqrt()) #convert to meters
    .rename('dist_to_water')
    .clip(fct_l0.geometry())
)

# Maximum distance 'distance_to_water' layer
print(distance_to_water.reduceRegion(ee.Reducer.max(), fct_l0.geometry(), 1000, maxPixels=1e9).getInfo())

thematic_map_2 = geemap.Map(center=fct_center, zoom=8)
thematic_map_2.add_basemap('SATELLITE')
thematic_map_2.addLayer(distance_to_water, vis_params_water, 'Permanent Water')
thematic_map_2.addLayer(distance_to_water.select('dist_to_water'), vis_params_dist_water, 'Distance to Water')

thematic_map_2

# Night Time Light

In [ ]:
# Nighttime lights (VIIRS DNB, annual composite)

# Data source: https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_DNB_MONTHLY_V1_VCMSLCFG

def get_night_time_lights(year, study_extent):
    '''Annual mean VIIRS radiance composite.'''
    start_date = ee.Date.fromYMD(year, 1, 1) # "2015", "january", "1"
    end_date = start_date.advance(1, 'year') # "2016", "january", "1"
    composite = (
        ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
        .filterDate(start_date, end_date)
        .select('avg_rad')
        .mean()
        .rename("ntl")
        .clip(study_extent)
    )
    return composite
# Apply the function to get night time lights for 2015, 2020, and 2025
ntl_2015 = get_night_time_lights(2015, fct_l0.geometry()) # Download night time lights for 2015
ntl_2020 = get_night_time_lights(2020, fct_l0.geometry()) # Download night time lights for 2020
ntl_2025 = get_night_time_lights(2025, fct_l0.geometry()) # Download night time lights for 2025

thematic_map_3 = geemap.Map(center=fct_center, zoom=8)
thematic_map_3.add_basemap('SATELLITE')
thematic_map_3.addLayer(ntl_2015, vis_params_ntl, 'Nighttime Lights 2015')
thematic_map_3.addLayer(ntl_2020, vis_params_ntl, 'Nighttime Lights 2020')
thematic_map_3.addLayer(ntl_2025, vis_params_ntl, 'Nighttime Lights 2025')
thematic_map_3

# Population Density

In [ ]:
# population density (CIESIN GPWv4.11)
# Data source: https://developers.google.com/earth-engine/datasets/catalog/CIESIN_GPWv411_population_density
# Closest available year to analysis baseline (2015/2020/2025)
gpw = ee.ImageCollection('CIESIN/GPWv411/GPW_Population_Density')

def get_population_density(year, extent):
    ''' Return the GPWv411 population density image closest to the given year.'''
    available_years = [2000, 2005, 2010, 2015, 2020]
    closest_year = min(available_years, key=lambda y: abs(y - year))
    image = ( 
        gpw.filter(ee.Filter.calendarRange(closest_year, closest_year, "year"))
        .first()
        .select('population_density')
        .rename("pop_density")
        .clip(extent)
        
    )
    return image
# Apply the function
pop_density_2015 = get_population_density(2015, fct_l0.geometry())
pop_density_2020 = get_population_density(2020, fct_l0.geometry())
pop_density_2022 = get_population_density(2025, fct_l0.geometry())

thematic_map_4 = geemap.Map(center=fct_center, zoom=8)
thematic_map_4.add_basemap('SATELLITE')
thematic_map_4.addLayer(pop_density_2015, vis_params_gpw, 'population density - 2015')
thematic_map_4.addLayer(pop_density_2020, vis_params_gpw, 'population density - 2020')
thematic_map_4.addLayer(pop_density_2022, vis_params_gpw, 'population density - 2022 (uses 2022 data)')
thematic_map_4
  

Land Cover

In [ ]:
# GLC_FCS30D Global Land cover
# Data source : https://gee-community-catalog.org/projects/glc_fcs/?h=glc+fcs30

# Reference : https://gee-community-catalog.org/tutorials/examples/glc_fcs30d_lulc/

glc_annual = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")

glc_mosaic = glc_annual.mosaic()

# Mosaic images and rename bands b1, b2 .... 2000, 2001, ....
# Each image in annual land cover data has 23 bands, one for each year from 2000 to 2022 (23 years)

glc_years = ee.List.sequence(2000, 2022).map(lambda y : ee.Number(y).format("%04d"))

glc_mosaic_renamed = glc_mosaic.rename(glc_years)

#print("GLC_FCS30D bands original:", glc_mosaic.bandNames().slice(0,5).getInfo(), "...")

#print("GLC_FCS30D bands renamed:", glc_mosaic_renamed.bandNames().slice(0,5).getInfo(), "...")



In [ ]:
# Classification scheme 
# (35 landcover  and 1 fill value)
ls_class_value = [
    10, 11, 12, 20, 51, 52, 61, 62, 71, 72, 81, 82, 91, 92, 120, 121, 122, 
    130, 140, 150, 152, 153, 181, 182, 183, 184, 185, 186, 187, 190, 200, 
    201, 202, 210, 220, 0]
# land cover class
glc_class_names = [
    'Rainfed_cropland', 'Herbaceous_cover_cropland', 'Tree_or_shrub_cover_cropland', 
    'Irrigated_cropland', 'Open_evergreen_broadleaved_forest', 'Closed_evergreen_broadleaved_forest', 
    'Open_deciduous_broadleaved_forest', 'Closed_deciduous_broadleaved_forest', 
    'Open_evergreen_needle_leaved_forest', 'Closed_evergreen_needle_leaved_forest', 
    'Open_deciduous_needle_leaved_forest', 'Closed_deciduous_needle_leaved_forest', 
    'Open_mixed_leaf_forest', 'Closed_mixed_leaf_forest', 'Shrubland', 'Evergreen_shrubland',
    'Deciduous_shrubland', 'Grassland', 'Lichens_and_mosses', 'Sparse_vegetation', 'Sparse_shrubland',
    'Sparse_herbaceous', 'Swamp', 'Marsh', 'Flooded_flat', 'Saline', 'Mangrove', 'Salt_marsh', 'Tidal_flat',
    'Impervious_surfaces', 'Bare_areas', 'Consolidated_bare_areas', 'Unconsolidated_bare_areas', 'Water_body', 
    'Permanent_ice_and_snow', 'Filled_value']

glc_class_colours = [
    '#ffff64', '#ffff64', '#ffff00', '#aaf0f0', '#4c7300', 
    '#006400', '#a8c800', '#00a000', '#005000', '#003c00',
    '#286400', '#285000', '#a0b432', '#788200', '#966400',
    '#964b00', '#966400', '#ffb432', '#ffdcd2', '#ffebaf',
    '#ffd278', '#ffebaf', '#00a884', '#73ffdf', '#9ebb3b', 
    '#828282', '#f57ab6', '#66cdab', '#444f89', '#c31400', 
    '#fff5d7', '#dcdcdc', '#fff5d7', '#0046c8', '#ffffff', '#ffffff']

    


In [ ]:
# visualize landcover for Abuja/FCT
lc_map = geemap.Map(center=fct_center, zoom=8)
lc_map.add_basemap('SATELLITE')
lc_map.addLayer(glc_mosaic_renamed.select("2015").clip(fct_l0.geometry()),{"palette" : glc_class_colours},"Land Cover - 2015")
lc_map.addLayer(glc_mosaic_renamed.select("2022").clip(fct_l0.geometry()),{"palette" : glc_class_colours},"Land Cover - 2022")
lc_map

# Combined Layers

In [ ]:
# Combine/stack layers
# 2022 predictor stack

predictor_stack = (
    elevation
    .addBands(slope)
    .addBands(distance_to_roads)
    .addBands(distance_to_water)
    .addBands(pop_density_2020)
    .addBands(ntl_2020)
    .addBands(ndvi)
    .addBands(ndwi)
    .addBands(ndbi)


)
#print("predictor bands:", predictor_stack.bandNames().getInfo())

# Extract only Urban class from landcover

In [ ]:
START_YEAR = 2015
END_YEAR = 2022
#print(START_YEAR)
#print(END_YEAR)
GLC_URBAN_CODE = 190

In [ ]:
# Extract only Urban class from landcover
def get_glc_urban_mask(year):
    '''Return binary urban (1) /non_urban (0) for a given year
    from the GLC_FCS30D dataset clipped to the AOI.'''
    band_name = str(year)
    year_image = glc_mosaic_renamed.select([band_name])
    urban_mask = year_image.eq(GLC_URBAN_CODE).rename("urban")
    return urban_mask. clip(fct_l0.geometry()).set({"year": year, "source" : "GLC_FCS30D"})

urban_by_year = {
    year: get_glc_urban_mask(year) for year in range(START_YEAR,END_YEAR + 1)
}
all_years = sorted(urban_by_year.keys())
#print("Urban time series (GLC_FCS30D only) covers:", all_years)


In [ ]:
urban_vis = {"min": 0, "max": 1, "palette": ["00000000", "d7301f"]}


Map = geemap.Map()
Map.add_basemap('SATELLITE')
Map.centerObject(fct_l0.geometry(), 10)


Map.addLayer(fct_l0.style(color="black", fillColor="ffffff00"), {}, "FCT_Boundary")

for year in [2015, 2018, 2022]:
    Map.addLayer(urban_by_year[year], urban_vis, f"urban {year}")
Map

In [ ]:
def compute_urban_area_Km2(urban_image,region = aoi, scale = 30):
    '''Return total urban area in km^2 for a binary urban image.'''
    area_image = urban_image.multiply(ee.Image.pixelArea())
    stats = area_image.reduceRegion(
        reducer = ee.Reducer.sum(),
        geometry = region,
        scale = scale,
        maxPixels = 1e10,
    
    )
    area_m2 = ee.Number(stats.get("urban"))
    return area_m2.divide(1e6) # convert m^2 to km^2
urban_area_records = []
for year in all_years:
    area_km2 = compute_urban_area_Km2(urban_by_year[year], scale = 30).getInfo()
    urban_area_records.append({"year":year, "urban_area_km2":area_km2})
    #print(f"{year}:{area_km2:.2f} Km2")

# To dataframe
urban_area_df = pd.DataFrame(urban_area_records)
#urban_area_df

In [ ]:
# Annual Expansion rate (km^2/year)
urban_area_df["expansion_km2"] = urban_area_df["urban_area_km2"].diff()
urban_area_df["expansion_km2"] = urban_area_df["urban_area_km2"].pct_change()*100
#urban_area_df


In [ ]:
import matplotlib.pyplot as plt
fig, ax1 = plt.subplots(figsize = (10,5))
ax1.plot(urban_area_df["year"], urban_area_df["urban_area_km2"], marker = "o", color ="firebrick")
ax1.set_xlabel("Year")
ax1.set_ylabel("urban area (km2)", color ="firebrick")
ax1.set_title("FCT/Abuja - Urban Extent Trend (2015-2022, GLC_FCS30D)")

ax2 = ax1.twinx()
ax2.bar(urban_area_df["year"], urban_area_df["expansion_km2"], alpha = 0.3, color = 'steelblue')
ax2.set_ylabel("Annual growth rate (%)", color ="steelblue")

plt.tight_layout()
plt.savefig("urban_growth_trend.png", dpi = 200)

In [ ]:
# Resampled stacked layers/images
predictor_stack_resampled = (predictor_stack.resample("bilinear")
                             .reproject(crs = predictor_stack.projection(), scale = 30))

In [ ]:
# Fetch Urban layers. Each pixel here is either 0 (non-urban) or 1 (urban)
urban_2015 = urban_by_year[2015]
urban_2022 = urban_by_year[END_YEAR]

# Four possible transitions
# 0 = Non_urban(2015) - Non_urban(2022)
# 1 = Non_urban(2015) - Urban(2022)
# 2 = Urban(2015)     - Non_urban(2022)
# 3 = Urban(2015)     - Urban(2022) 

# Encode all 4 transition combinations in a single raster
transition_15_22 = urban_2015.multiply(2).add(urban_2022).rename("transition")

# Check unique pixel values
transition_hist = transition_15_22.reduceRegion(
    reducer = ee.Reducer.frequencyHistogram(),
    geometry =aoi, 
    scale = 10,
    maxPixels=1e13
)

unique_transition_values = ee.Dictionary(transition_hist.get('transition')).keys().getInfo()

#print(unique_transition_values)


In [ ]:
# Keep two transitions: code 0 and 1
non_urban_2015 = urban_2015.eq(0) # removing the urban areas that are equal to 0 leaving non_urban areas that are equal to 1
# The transition code (0 or 1) is already the label
change_label = transition_15_22.updateMask(non_urban_2015).rename("label") 

#print("Transition codes kept: 0 = stable non_urban, 1 = new urban (2015 - 2022)")

In [ ]:
# Area of 4 transition classes
transition_area = ee.Image.pixelArea().addBands(transition_15_22).reduceRegion(
    reducer = ee.Reducer.sum().group(groupField = 1, groupName = "transition"),
    geometry = aoi,
    scale = 30,
    maxPixels=1e10
)
transition_groups = ee.List(transition_area.get("groups")).getInfo()
transition_names = {
    0: "Non-urban - Non-urban",
    1: "Non-urban - Urban (growth)",
    2: "Urban - Non-urban (loss)",
    3: "Urban - Urban (stable)"
}
for group in transition_groups:
    code_val = int(group["transition"])
    area_km2 = group["sum"]/ 1e6
   # print(f"{transition_names[code_val]:32s}: {area_km2:9.2f} km^2")

In [ ]:
# Final training image: predictor stack + land cover transition layer
training_images = (predictor_stack
                  .addBands(change_label)
                  .updateMask(non_urban_2015))
#print(training_images.bandNames().getInfo())

In [ ]:
# Maps to visualise ML results
# Center coordinates to show map
ml_map = geemap.Map(center = fct_center,zoom = 8)
ml_map.add_basemap("SATELLITE")




In [ ]:
# Stratified random sample
NUM_POINTS_PER_CLASS  = 1500

samples = training_images.stratifiedSample(
    numPoints = NUM_POINTS_PER_CLASS,
    classBand = "label",
    region = aoi,
    scale = 30,
    seed = 42, 
    geometries = True
)
#print("Total training samples:", samples.size().getInfo())
#print("Class distribution:", samples.aggregate_histogram("label").getInfo())

In [ ]:
 #To pandas DataFrame from FC
def fc_to_df(fc):
    features = fc.getInfo()["features"]
    rows = [f["properties"] for f in features]
    return pd.DataFrame(rows)

samples_df = fc_to_df(samples)
samples_df = samples_df.dropna()
#print(samples_df.shape)
#display(samples_df.head())

In [ ]:
# Predictors 
#print(samples_df.columns.tolist())
PREDICTOR_VARS = [col for col in samples_df.columns.tolist() if col != "label"]
#PREDICTOR_VARS

In [ ]:
import numpy as np
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Check Multi-collinearity
X_raw = samples_df[PREDICTOR_VARS].copy()

# Correlation
corr_table = X_raw.corr()
#print("Correlation Table")
#display(corr_table.round(3)) 


plt.figure(figsize=(14, 12))
sns.heatmap(corr_table, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Matrix Heatmap", fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
# Check VIF
# Constant (intercept) for VIF calculation
X_vif = sm.add_constant(X_raw)
vif_df = pd.DataFrame()
vif_df['Variable'] = X_vif.columns
vif_df['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

# Intercept 
vif_df = vif_df[vif_df['Variable'] != 'const'].sort_values('VIF', ascending=False)
#print("Variance Inflation Factors (VIF)")
#display(vif_df) 

In [ ]:
# Split sample into train/test. 75% training and 25% testing
samples_split = samples.randomColumn("random", seed = 42)
train_samples = samples_split.filter(ee.Filter.lt("random", 0.75))
test_samples = samples_split.filter(ee.Filter.gte("random", 0.75))

#print("Train samples :", train_samples.size().getInfo())
#print("Test samples :", test_samples.size().getInfo())


In [ ]:
# Visualise the training and testing samples
training_style = {"color":"blue","pointSize":3}
testing_style = {"color":"red","pointSize":3}

training_layer = train_samples.style(**training_style)
testing_layer = train_samples.style(**testing_style)
ml_map.addLayer(fct_l0.style(color = "black", fillColor = "00000000"), {}, "FCT Boundary")
ml_map.addLayer(training_layer, {},"Training Samples")
ml_map.addLayer(testing_layer, {}, "Testing Samples")

ml_map



## Random Forest (Earth Engine)

In [ ]:
#training_image.bandNames().getInfo()

In [ ]:
# print(training_image,bandNames).getInfo()[:-1]
features_cols = training_image.bandNames().getInfo()[:-1]
#features_cols

In [ ]:
# Initialise and train Random Forest model
ee_classifier = ee.Classifier.smileRandomForest(
    numberOfTrees = 500,
    minLeafPopulation = 5,
    seed = 42,
).setOutputMode("PROBABILITY")

ee_classifier_eval = ee_classifier.train(
    features = train_samples,
    classProperty = "label",
    inputProperties = features_cols,
)
#print("Random Forest trained on the train split")

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
# classify the held-out  test portion
test_classified = test_samples.classify(ee_classifier_eval, outputName = "suitability")
test_df = fc_to_df(test_classified).dropna()
y_true = test_df["label"].astype(int)
y_prob = test_df["suitability"]
y_pred = (y_prob >= 0.5).astype(int)

#print("Held-out test-split result:")
#print("Accuracy:", accuracy_score(y_true, y_pred))
#print("ROC-AUC :", roc_auc_score(y_true, y_prob))
#print()
#print(classification_report(y_true, y_pred))

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize = (5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
            xticklabels=["No Change", "New Urban"],
            yticklabels=["No Change", "New Urban"])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=200)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
# ROC curve
fpr, tpr,_ =roc_curve(y_true, y_prob)
plt.figure(figsize = (5,5))
plt.plot(fpr, tpr, color = "firebrick", label = f"AUC = {roc_auc_score(y_true,y_prob):.3f}")
plt.plot([0, 1],[0, 1], linestyle = "--", color = "gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve-- Held-out Test Split (2015 -> 2022)")
plt.legend()
plt.tight_layout()
plt.savefig("roc_curve.png", dpi = 200)
plt.show()

In [ ]:
# Retrain on the Full sample set for the production model
ee_classifier_trained = ee_classifier.train(
    features = samples,
    classProperty= "label",
    inputProperties = features_cols,
)
#print("Final random forest retrained on the full 2015 - 2022 sample set.")

In [ ]:
# Feature Importance
importance_dict = ee_classifier_trained.explain().getInfo()["importance"]
importance_df = pd.DataFrame(
    list(importance_dict.items()), columns=["feature", "importance"]
).sort_values("importance", ascending=False)

plt.figure(figsize = (8,5))
sns.barplot(data = importance_df, x = 'importance', y = 'feature', color = 'firebrick')

plt.title("Feature Importance -- Driver of Urban Growth (2015->2022)", fontsize=14, pad=15)


plt.xlabel("Relative Importance", fontsize=12, labelpad=10)
plt.ylabel("Feature", fontsize=12, labelpad=10)


plt.tick_params(axis='both', labelsize=10)

plt.tight_layout()
plt.savefig("feature_importance.png", dpi = 200)
plt.show()


## Suitability Map (Apply trained model to every pixel)

In [ ]:
growth_suitability = predictor_stack.select(features_cols).classify(
    ee_classifier_trained
).rename("suitability")

In [ ]:
Suitability_map = geemap.Map(center = fct_center, zoom =10)
Suitability_map.add_basemap("SATELLITE")
suitability_vis = {"min":0, "max":1, "palette":["ffffcc", "fd8d3c", "800026"]}
Suitability_map.addLayer(fct_l0.style(color = "black",fillColor = "00000000"), {},"FCT Boundary")
Suitability_map.addLayer(growth_suitability,suitability_vis, "Urban Growth Suitability")
Suitability_map.add_colorbar(suitability_vis, label = "Probability of becoming urban")



Suitability_map

In [ ]:
non_urban_2022 = urban_2022.eq(0)
candidate_suitability = growth_suitability.updateMask(non_urban_2022)
Suitability_map2 = geemap.Map(center = fct_center, zoom =10)
Suitability_map2.add_basemap("SATELLITE")
Suitability_map2.addLayer(fct_l0.style(color = "black",fillColor = "00000000"), {},"FCT Boundary")
Suitability_map2.addLayer(candidate_suitability,suitability_vis, "Suitability")
Suitability_map2.addLayerControl()

Suitability_map2
